In [58]:
import numpy as np
import pandas_ta as ta
import pandas as pd
from datetime import time
from matplotlib import pyplot as plt
#can plot candlestick charts
import mplfinance as mpf

NQ = pd.read_csv("/Users/teddieshi/Documents/vscode/backtest/algotrading/data/1mNQData.csv")

NQ = NQ.rename(columns = {
    "timestamp_utc" : "Date",
    "open" : "Open",
    "high" : "High",
    "low" : "Low",
    "close" : "Close",
    "volume" : "Volume",
    "risk_free_rate_annual" : "RiskFreeRate"
})

NQ["Date"] = pd.to_datetime(NQ["Date"]).dt.tz_convert("America/New_York")
NQ = NQ.set_index("Date")



In [59]:
#NY session & VWAP calculations
session = NQ.between_time("09:30", "16:00", inclusive= "left").copy()
session["TypicalPrice"] = (session["High"] + session["Low"] + session["Close"]) / 3 #Approximate price per candle
session["PriceVolume"] = (session["TypicalPrice"] * session["Volume"]) 

key = session.index.normalize() #change each timestamp to midnight so all candles on the same date receive the save key

session["CumulativeVolume"] = session.groupby(key)["Volume"].cumsum() #creates a new column that represents the accumulated volume on the day
session["CumulativePriceVolume"] = session.groupby(key)["PriceVolume"].cumsum() #creates a new column that represents the accumulated price volume on the day
session["VWAP"] = session["CumulativePriceVolume"]/session["CumulativeVolume"]
session["PrevVWAP"] = session["VWAP"].shift(1)

session["PriceAboveVWAP"] = session["Close"] > session["VWAP"]


In [ ]:
VWAPpullback = False
enter_trade = False
trade_entered = False
candle_low = float("inf")
TP_ratio = 2

for i, row in session.iterrows():
    if i.time() >= time(9,45) and row.VWAP > row.PrevVWAP:
        if row.Low <= row.VWAP:
            VWAPpullback = True
            while row.Low < candle_low:
                candle_low = row.Low
        if VWAPpullback:
            if row.Close > row.VWAP:
                enter_trade = True
        if enter_trade and trade_entered == False:
            trade_entered = True
            SL = candle_low
            TP = candle_low * TP_ratio
            